# 미국 뉴스 거르기

`news_collector.ipynb`가 쌓은 미국 뉴스에서 **뉴스가 아닌 글을 규칙으로 걸러 표시**하고, 기사마다 정보 추출(`news_extractor.ipynb`)에 **무엇을 넣을지**(본문, 제목·요약, 제외)를 정한다. 규칙과 이유는 `graphDB.md` 4.4를 따른다.

- 이 노트북은 **외부 서버에 요청하지 않는다.** DB를 읽기만 하므로 몇 번을 다시 실행해도 된다.
- **지우지 않고 표시한다.** 모든 기사에 한 줄씩 남기고, 제외한 기사에는 사유와 걸린 규칙을 적는다.
- 판단 근거는 RSS 항목의 **주소, 분류 태그, 제목**이다. 일부 규칙은 RSS 요약이나 본문 첫 줄도 본다.
- 결과를 LLM 추출과 따로 저장한다. 거르기 규칙은 자주 고치지만 LLM 추출은 다시 돌리기 부담스럽기 때문이다. 규칙을 고치면 `FILTER_VERSION`을 바꾸고 다시 실행한다.
- 수집기와 공유 DB에는 아무것도 쓰지 않는다.

**입력·출력**

| 구분 | 위치 | 내용 |
|---|---|---|
| 입력 | 공유 DB의 `articles` 표 (`.env`의 `NEWS_DB_READ_URL`, 읽기 전용 계정) | 접속이 안 되면(윈도우 PC와 다른 네트워크) `source/news/snapshot_*/news.sqlite3` 중 가장 최근 것을 읽는다 |
| 출력 | `source/news/filter/article_filter.csv` | 기사 하나에 한 줄 (UTF-8 BOM CSV) |
| 출력 | `source/news/filter/filter_meta.json` | 읽은 곳, 실행 시각, 규칙 버전, 건수 |

**`use` 열** (추출에 무엇을 넣을지)

| 값 | 뜻 |
|---|---|
| `body` | 본문을 쓴다 |
| `summary` | 제목과 RSS 요약만 쓴다. 본문을 받지 못했거나(봇 차단 등) 유료벽에 잘린 기사 |
| `exclude` | 쓰지 않는다. `reason`에 사유가 있다 |
| `wait` | 본문을 아직 받지 않았다(`pending`). 다음 실행에서 다시 정한다 |

**걸리는 시간**: 몇 초.

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `news_collector`(윈도우 PC) → **`news_filter`** → `news_extractor` |
| 외부 요청·비용 | 없음. 공유 DB를 읽기 전용으로 읽거나, 안 되면 스냅샷을 읽는다 |
| 다시 실행 | 언제든 된다. 규칙을 고치면 `FILTER_VERSION`을 바꾼다 |
| 처리 순서 | 설정 → 기사 읽기 → 본문 상태 → 뉴스가 아닌 글 규칙 → 오래된 기사·중복 → 보도자료 표시 → 판단·저장 → 확인 |

## 1. 설정

In [1]:
# ── 설정: 읽을 곳(공유 DB 또는 최근 스냅샷), 규칙 버전, 출력 폴더 ──
import datetime as dt
import html
import json
import re
from pathlib import Path
from urllib.parse import urlsplit

import pandas as pd
import sqlalchemy as sa

pd.set_option('display.max_colwidth', 90)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 200)

# 이 노트북은 graph_db 폴더에서 실행한다.
ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'
NEWS_DIR = ROOT / 'source' / 'news'
OUT = NEWS_DIR / 'filter'                        # 이 노트북이 만드는 파일
OUT.mkdir(parents=True, exist_ok=True)

FILTER_VERSION = '2026-11-11'                    # 규칙을 고치면 날짜를 바꾼다

# 읽을 곳. 'auto'면 공유 DB(NEWS_DB_READ_URL)를 먼저 시도하고, 접속이 안 되면 가장 최근 스냅샷을 읽는다.
# 특정 스냅샷을 읽으려면 경로를 넣는다. 예: 'source/news/snapshot_20261112_1127/news.sqlite3'
NEWS_SOURCE = 'source/news/snapshot_20261112_1127/news.sqlite3'
CONNECT_TIMEOUT_SEC = 5


def env_value(name):
    """환경변수 → 프로젝트 루트 .env → graph_db/.env 순서로 값을 찾는다. 없으면 None."""
    import os
    if os.environ.get(name):
        return os.environ[name]
    for env_file in (ROOT.parent / '.env', ROOT / '.env'):
        if not env_file.exists():
            continue
        for line in env_file.read_text(encoding='utf-8-sig').splitlines():
            key, _, value = line.partition('=')
            if key.strip() == name and value.strip():
                return value.strip().strip('"\'')
    return None


def mask(url):
    """DB 주소의 계정·비밀번호를 가린다."""
    return re.sub(r'//[^@]+@', '//***@', url)


def open_source():
    """읽을 DB를 연다. (engine, 화면·기록용 이름)을 돌려준다."""
    if NEWS_SOURCE != 'auto':
        path = ROOT / NEWS_SOURCE
        assert path.exists(), f'스냅샷이 없다: {NEWS_SOURCE}'
        return sa.create_engine(f'sqlite:///{path.as_posix()}'), path.relative_to(ROOT).as_posix()
    url = env_value('NEWS_DB_READ_URL')
    if url:
        try:
            engine = sa.create_engine(url, connect_args={'connect_timeout': CONNECT_TIMEOUT_SEC})
            with engine.connect() as c:
                c.execute(sa.text('SELECT 1'))
            return engine, mask(url)
        except Exception as e:                   # 오류 문구에 주소가 들어갈 수 있어 종류만 보인다
            print('공유 DB에 접속하지 못해 스냅샷을 읽는다:', type(e).__name__)
    snapshots = sorted(NEWS_DIR.glob('snapshot_*/news.sqlite3'))
    assert snapshots, '읽을 스냅샷이 없다.'
    return sa.create_engine(f'sqlite:///{snapshots[-1].as_posix()}'), snapshots[-1].relative_to(ROOT).as_posix()

## 2. 기사 읽기

`articles` 표에서 판단에 필요한 열만 읽는다. 분류 태그(`categories`)는 RSS의 태그 목록(JSON)이고, 요약(`summary_html`)은 RSS의 HTML 그대로라 글자만 남겨서 본다. 제목에도 `&amp;`, `<i>` 같은 표시가 섞여 있어 같이 정리한다.

In [2]:
# ── 2. 기사 읽기: 판단에 필요한 열만 ──
engine, SOURCE_NAME = open_source()
COLUMNS = ['article_id', 'source', 'url', 'title', 'summary_html', 'categories',
           'published_utc', 'first_seen_utc', 'body_status', 'body_text', 'body_flags']
with engine.connect() as c:
    df = pd.read_sql(sa.text(f"SELECT {', '.join(COLUMNS)} FROM articles"), c)
print('읽은 곳:', SOURCE_NAME)
print('기사', len(df), '건 /', df['source'].nunique(), '개 매체 /',
      df['published_utc'].min()[:10], '~', df['published_utc'].max()[:10])


def plain(h):
    """HTML → 글자만(엔티티 풀고 태그 지우고 공백 정리)."""
    return re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', html.unescape(h or ''))).strip()


def tag_list(raw):
    """categories(JSON 목록) → 태그 목록. 읽을 수 없으면 빈 목록."""
    try:
        return [str(t).strip() for t in json.loads(raw or '[]') if str(t).strip()]
    except (TypeError, ValueError):
        return []


df['path'] = df['url'].map(lambda u: urlsplit(u or '').path.lower())
df['tag_text'] = df['categories'].map(lambda raw: '|'.join(t.lower() for t in tag_list(raw)))
df['title_text'] = df['title'].map(plain)
df['summary_text'] = df['summary_html'].map(plain)
df['body_head'] = df['body_text'].fillna('').str[:300]

df['body_status'].value_counts().rename('기사 수').to_frame()

읽은 곳: source/news/snapshot_20261112_1127/news.sqlite3
기사 2725 건 / 62 개 매체 / 2020-06-04 ~ 2026-11-12


,기사 수
body_status,
ok,2401
skipped_rule,104
blocked,87
short,71
paywalled,28
suspect,26
gone,3
empty,3
sponsored,2


## 3. 본문 상태 → 추출에 넣을 것

수집기가 본문을 받으며 남긴 상태(`body_status`, `news_collector.ipynb` 맨 위 표)에 따라 정한다.

| 본문 상태 | 추출에 쓰는 것 | 이유 |
|---|---|---|
| `ok`, `short`, `suspect` | 본문 | `short`는 원래 짧은 소식 기사가 많다. `suspect`는 확인해 보니 대부분 제목 단어가 본문에 적게 나온 오탐이었다 |
| `blocked`, `paywalled`, `empty`, `expired`, `error`, `robots` | 제목과 RSS 요약 | 막힌 기사는 RSS 제목·요약만 남긴다(4.1). 유료벽 기사는 본문을 쓰지 않는다(4.4). RSS 요약은 언론사가 공개로 배포한 것이다 |
| `sponsored`, `skipped_rule`, `gone` | 쓰지 않음 | 후원 기사, 수집기 주소 규칙에 걸린 기사(유료·후원·영상·자료 페이지), 내려간 기사 |
| `pending` | 미룸 | 아직 본문을 받지 않았다 |

정하지 않은 상태가 나오면 멈춘다. 수집기에 새 상태가 생기면 이 표에 넣는다.

In [3]:
# ── 3. 본문 상태 → 추출에 넣을 것(body / summary / wait) ──
STATUS_INPUT = {
    'ok': 'body', 'short': 'body', 'suspect': 'body',
    'blocked': 'summary', 'paywalled': 'summary', 'empty': 'summary',
    'expired': 'summary', 'error': 'summary', 'robots': 'summary',
    'sponsored': 'exclude', 'skipped_rule': 'exclude', 'gone': 'exclude',
    'pending': 'wait',
}
unknown = set(df['body_status'].fillna('(없음)')) - set(STATUS_INPUT)
assert not unknown, f'처리 방식을 정하지 않은 본문 상태: {unknown}'
df['status_input'] = df['body_status'].map(STATUS_INPUT)
df['status_input'].value_counts().rename('기사 수').to_frame()

,기사 수
status_input,
body,2498
summary,118
exclude,109


## 4. 뉴스가 아닌 글

RSS 항목의 주소·분류 태그·제목(일부는 요약·본문 첫 줄)으로 판단한다. 분류 태그가 없는 매체(CNBC, CBS, LA Times, Industry Dive 등)는 주소와 제목으로만 판단한다. 한 기사가 여러 규칙에 걸리면 표에서 위에 있는 사유를 쓰고, 걸린 규칙은 모두 `rule_matched`에 남긴다.

| 사유 (`reason`) | 판단 근거 | 예 |
|---|---|---|
| `sponsored` 후원·광고 | 태그 Sponsored·White Papers·Partner Content, 요약 첫머리 "[Sponsored]", 주소 `supplier-spotlight` | EE Times 후원 기사, Breaking Defense Sponsored Post |
| `media` 영상·팟캐스트·사진 | 주소 `/video/`·`/podcasts/`·`photo-gallery`, 태그 Videos·Podcasts·Galleries, 제목 "Podcast:"·"In pictures:"·"… in photos" | Glossy Podcast, 행사 사진 모음 |
| `data_page` 시세·자료·라이브 페이지 | 주소 `/price-of-`·`live-updates`·`/industry-statistics/`·`/document/` | Fortune 유가 페이지, CNBC 시장 라이브, OGJ 통계 |
| `lifestyle` 생활·커리어 | CNBC `/make-it/`·`/select/`, Fortune 태그 Success·Workplace Culture·Future of Work·C-Suite·Travel & Leisure, 쇼핑 추천 제목 | 면접 요령, 카드 신청 요령, CEO 일화 |
| `opinion` 외부 기고 | 태그 Opinion·Commentary·Reader Forum, 주소 `/opinion/`·`/reader-forum/` | 업체 임원 기고가 많고 사실보다 주장이 중심이다 |
| `promo` 매체 홍보·행사 안내 | 주소 `/newsletters/`, 제목 "Register:"·"webinar"·"free eBook"·"Get the … newsletter" 등, 본문이 "Event venue:"로 시작하는 행사 안내 | 뉴스레터 묶음, 시상 접수 안내 |
| `people_roundup` 인사 동정 모음 | 제목 "People Moves"·"People on the Move"·"Jobs roundup" | 개별 CEO 교체 기사는 남긴다 |
| `non_english` 영어가 아닌 기사 | Mobile World Live `/spanish/` | 스페인 통신 소식이 대부분이다 |

**거르지 않는 것**
- 매체가 여러 뜻으로 쓰는 태그는 근거로 쓰지 않는다. Defense News·Fortune의 `Newsletters`, Fortune의 상위 태그 `Lifestyle`은 일반 기사에도 붙는다.
- 주간 모음 기사(Eurobites, Week in Review 등)는 실제 소식이 많아 남긴다.
- 정치·사건 기사, 일반 주소에 있는 개인 재무·종목 추천 글은 남긴다. 주소·제목만으로는 경제 기사와 가르기 어려워서, 추출 단계에서 LLM이 관련 정보가 없다고 판단하게 한다.

In [4]:
# ── 4. 뉴스가 아닌 글: 주소·태그·제목 규칙(사유, 볼 곳, 정규식, 적용 매체) ──
# (사유, 볼 곳, 정규식, 적용 매체). 매체가 None이면 모든 매체. 대소문자는 가리지 않는다.
# 태그는 소문자로 바꿔 '|'로 이어 붙인 글자(tag_text)에서 찾는다.
RULES = [
    ('sponsored',      'tag',     r'sponsor|white ?papers?|partner content|focus on partners', None),
    ('sponsored',      'summary', r'^\[sponsored\]', None),
    ('sponsored',      'path',    r'supplier-spotlight', None),
    ('media',          'path',    r'/videos?/|/podcasts?/|photo-gallery|/galler', None),
    ('media',          'tag',     r'(^|\|)(videos?|podcasts?|tv & podcasts|galleries)(\||$)', None),
    ('media',          'title',   r'^podcast:|^in pictures:| in photos$|^in the know \d+', None),
    ('data_page',      'path',    r'/price-of-|live-updates|/industry-statistics/|/document/', None),
    ('lifestyle',      'path',    r'^/make-it/|^/select/', ['CNBC']),
    ('lifestyle',      'tag',     r'(^|\|)(success|workplace culture|future of work|c-suite|travel & leisure)(\||$)', ['Fortune']),
    ('lifestyle',      'title',   r'best .*gifts?\b|gift sets', None),
    ('opinion',        'tag',     r'(^|\|)(opinion|commentary|reader forum)(\||$)', None),
    ('opinion',        'path',    r'/opinion/|/reader-forum/', None),
    ('promo',          'path',    r'^/newsletters/', None),
    ('promo',          'title',   r'^register:|final call for entries|webinar|free ebook|^get the .* newsletter', None),
    ('promo',          'body',    r'^event venue:', None),
    ('people_roundup', 'title',   r'^people moves|^people on the move|^jobs roundup', None),
    ('non_english',    'path',    r'^/spanish/', ['Mobile World Live']),
]
REASON_KO = {
    'sponsored': '후원·광고', 'media': '영상·팟캐스트·사진', 'data_page': '시세·자료·라이브 페이지',
    'lifestyle': '생활·커리어', 'opinion': '외부 기고', 'promo': '매체 홍보·행사 안내',
    'people_roundup': '인사 동정 모음', 'non_english': '영어가 아닌 기사',
    'stale': '수집 때 이미 오래된 기사', 'duplicate': '중복', 'status': '본문 상태(수집 단계에서 쓰지 않기로 함)',
}
FIELDS = {'path': 'path', 'tag': 'tag_text', 'title': 'title_text', 'summary': 'summary_text', 'body': 'body_head'}
COMPILED = [(reason, FIELDS[field], re.compile(pattern, re.I), set(sources) if sources else None, f'{field}:{pattern}')
            for reason, field, pattern, sources in RULES]


def rule_hits(row):
    """이 기사에 걸린 (사유, 규칙) 목록. RULES 순서대로."""
    return [(reason, label) for reason, col, rx, sources, label in COMPILED
            if (sources is None or row['source'] in sources) and rx.search(row[col] or '')]


hits = df.apply(rule_hits, axis=1)
df['rule_reason'] = hits.map(lambda h: h[0][0] if h else '')
df['rule_matched'] = hits.map(lambda h: ' ; '.join(dict.fromkeys(label for _, label in h)))

# 규칙별로 걸린 기사 수 (한 기사가 여러 규칙에 걸릴 수 있다)
per_rule = pd.Series([label for h in hits for _, label in h]).value_counts()
pd.DataFrame({'기사 수': [per_rule.get(c[4], 0) for c in COMPILED]},
             index=pd.MultiIndex.from_tuples([(c[0], c[4]) for c in COMPILED], names=['사유', '규칙']))

기사 수
사유             규칙                                                                                       
sponsored      tag:sponsor|white ?papers?|partner content|focus on partners                           31
               summary:^\[sponsored\]                                                                  5
               path:supplier-spotlight                                                                 2
media          path:/videos?/|/podcasts?/|photo-gallery|/galler                                       27
               tag:(^|\|)(videos?|podcasts?|tv & podcasts|galleries)(\||$)                            19
               title:^podcast:|^in pictures:| in photos$|^in the know \d+                              4
data_page      path:/price-of-|live-updates|/industry-statistics/|/document/                          27
lifestyle      path:^/make-it/|^/select/                                                               8
               tag:(^|\|)(success|workplace culture|future of work|c-suite|travel & leisure)(\||$)    20
               title:best .*gifts?\b|gift sets                                                         1
opinion        tag:(^|\|)(opinion|commentary|reader forum)(\||$)                                      22
               path:/opinion/|/reader-forum/                                                           2
promo          path:^/newsletters/                                                                     8
               title:^register:|final call for entries|webinar|free ebook|^get the .* newsletter       6
               body:^event venue:                                                                      1
people_roundup title:^people moves|^people on the move|^jobs roundup                                   4
non_english    path:^/spanish/                                                                        10

## 5. 오래된 기사와 중복

**오래된 기사**: 처음 수집한 시각에 이미 게시일에서 90일이 넘은 기사는 쓰지 않는다. 피드에 다시 올라온 옛 글이다(예: FreightWaves 2021년 기사). 수집을 시작할 때 피드에 남아 있던 최근 기사(한두 달 전)는 쓴다. 관계의 시점은 게시일로 남긴다(#17).

**중복**: 같은 매체에 같은 제목으로 다시 올라온 기사는 먼저 나온 것만 남긴다(주소가 달라 기사 ID가 다르게 저장된 경우). 제목은 소문자로 바꾸고 문장부호를 빼고 비교한다. 다른 매체끼리 같은 사건을 다룬 기사는 각각 남긴다.

In [5]:
# ── 5. 오래된 기사: 수집 때 이미 게시일에서 90일이 넘은 기사 ──
STALE_DAYS = 90
age_days = (pd.to_datetime(df['first_seen_utc'], utc=True) - pd.to_datetime(df['published_utc'], utc=True)).dt.days
df['stale'] = age_days > STALE_DAYS
print(f'수집 때 이미 {STALE_DAYS}일 넘은 기사', int(df['stale'].sum()), '건')
display(df.loc[df['stale'], ['source', 'published_utc', 'first_seen_utc', 'title_text']].sort_values('published_utc'))

수집 때 이미 90일 넘은 기사 23 건


,source,published_utc,first_seen_utc,title_text
1909,CBS News,2020-06-04T20:58:00+00:00,2026-11-10T00:21:17+00:00,Nearly one in five say coronavirus has caused financial hardship
226,FreightWaves,2021-05-27T21:01:00+00:00,2026-11-10T00:21:50+00:00,Project44 acquires ClearMetal to strengthen predictive tools
434,FreightWaves,2021-06-27T18:00:00+00:00,2026-11-10T00:21:50+00:00,Seasonality pushing rejections and rates higher ahead of the Fourth
912,FreightWaves,2021-08-12T20:10:00+00:00,2026-11-10T00:21:50+00:00,Warehouse cramming is about to begin — Freightonomics
264,FreightWaves,2021-12-13T22:15:00+00:00,2026-11-10T00:21:50+00:00,Exclusive: Central Freight Lines to shut down after 96 years
1297,FreightWaves,2022-05-12T15:46:28+00:00,2026-11-10T00:21:50+00:00,Why the Northeast is quietly running out of diesel
1377,FreightWaves,2022-05-17T20:12:32+00:00,2026-11-10T00:21:50+00:00,The ‘ingenious strategy’ behind most truckers’ least favorite week of the year: Intern...
2520,CBS News,2025-08-04T12:12:00+00:00,2026-11-10T16:18:34+00:00,"Tracking cost of cars, trucks, groceries as Trump announces new tariffs"
2724,Fortune,2026-03-14T15:01:27+00:00,2026-11-10T00:21:14+00:00,Mary Barra still responds to ‘every single letter’ she gets by hand despite running $6...
2375,CNBC,2026-05-24T12:52:28+00:00,2026-11-11T10:23:48+00:00,Top Wall Street analysts like these dividend stocks for steady income


In [6]:
# ── 5. 중복: 같은 매체의 같은 제목은 먼저 나온 것만 쓴다 ──
df['title_key'] = df['title_text'].str.lower().str.replace(r'\W+', ' ', regex=True).str.strip()
df = df.sort_values(['published_utc', 'first_seen_utc'], na_position='last').reset_index(drop=True)
first_id = df.groupby(['source', 'title_key'])['article_id'].transform('first')
df['duplicate_of'] = ''
is_dup = (first_id != df['article_id']) & (df['title_key'] != '')
df.loc[is_dup, 'duplicate_of'] = first_id[is_dup]
print('중복', int(is_dup.sum()), '건')
df.loc[is_dup, ['source', 'title_text', 'body_status']]

중복 9 건


,source,title_text,body_status
47,CNBC,Top Wall Street analysts like these 3 dividend stocks for steady income,ok
241,CNBC,Top Wall Street analysts prefer these 3 dividend stocks for stable income,ok
1265,Maritime Executive,Kongsberg Maritime Unveils New Logo and Tagline,ok
1524,Recycling Today,The metal recycling industry’s evolution,blocked
1791,Automotive World,Iveco Group launches Daily 4×4 and Daily CNG at Fenatran,skipped_rule
1796,Automotive World,Li Auto takes right-hand-drive Li Mega to Southeast Asia,skipped_rule
1845,Electronics Weekly,Electronica – The latest news roundup,ok
2145,Defense News,Hegseth says impeachment filing against him is ‘sabotage’ and political ‘farce’,skipped_rule
2710,CBS News,"Nvidia's board preparing to oust CEO Jensen Huang, sources say",skipped_rule


## 6. 보도자료 표시

거르지 않고 표시만 한다. 관계를 검증할 때(#18) 기업 발표라는 근거의 무게를 다르게 둘 수 있다(4.5).

- 수집기가 본문 끝의 "Source: 회사명" 줄을 지우며 남긴 `body_flags.press_release_from`
- Maritime Executive 본문 첫 줄의 `[By: 회사명]`
- 태그 `Press Releases` (회사명은 모른다)

In [7]:
# ── 6. 보도자료 표시(거르지 않는다) ──
BY_LINE = re.compile(r'^\[By:\s*([^\]]+)\]')


def press_release(row):
    """(보도자료인지, 낸 곳). 낸 곳을 모르면 빈 글자."""
    try:
        flags = json.loads(row['body_flags']) if row['body_flags'] else {}
    except (TypeError, ValueError):
        flags = {}
    if flags.get('press_release_from'):
        return True, flags['press_release_from']
    m = BY_LINE.match(row['body_head'])
    if m:
        return True, m.group(1).strip()
    if re.search(r'(^|\|)press releases?(\||$)', row['tag_text']):
        return True, ''
    return False, ''


df[['press_release', 'press_release_from']] = df.apply(press_release, axis=1, result_type='expand')
print('보도자료', int(df['press_release'].sum()), '건')
df.loc[df['press_release']].groupby('source').size().sort_values(ascending=False).rename('보도자료').to_frame().T

보도자료

 55 건


source,Automotive World,Maritime Executive,Insurance Journal,gCaptain,Chemical Engineering,EE Times,FoodNavigator-USA,Offshore Energy,Splash247,The Robot Report,electrive
보도자료,37,7,2,2,1,1,1,1,1,1,1


## 7. 판단과 저장

순서대로 본다. 앞에서 정해지면 뒤는 보지 않는다.

1. 4절 규칙에 걸리면 `exclude` (사유는 규칙의 사유)
2. 5절 오래된 기사면 `exclude` (`stale`)
3. 5절 중복이면 `exclude` (`duplicate`)
4. 3절 본문 상태가 쓰지 않는 상태면 `exclude` (`status`)
5. 나머지는 3절 본문 상태에 따라 `body`, `summary`, `wait`

In [8]:
# ── 7. 판단과 저장: 규칙 → 오래됨 → 중복 → 본문 상태 순서로 use와 사유를 정한다 ──
def decide(row):
    """(use, reason)"""
    if row['rule_reason']:
        return 'exclude', row['rule_reason']
    if row['stale']:
        return 'exclude', 'stale'
    if row['duplicate_of']:
        return 'exclude', 'duplicate'
    if row['status_input'] == 'exclude':
        return 'exclude', 'status'
    return row['status_input'], ''


df[['use', 'reason']] = df.apply(decide, axis=1, result_type='expand')
df['reason_ko'] = df['reason'].map(REASON_KO).fillna('')
df['filter_version'] = FILTER_VERSION

OUT_COLUMNS = ['article_id', 'source', 'published_utc', 'title_text', 'url', 'body_status',
               'use', 'reason', 'reason_ko', 'rule_matched', 'duplicate_of',
               'press_release', 'press_release_from', 'filter_version']
result = df[OUT_COLUMNS].rename(columns={'title_text': 'title'})
result.to_csv(OUT / 'article_filter.csv', index=False, encoding='utf-8-sig')

meta = dict(filter_version=FILTER_VERSION, source=SOURCE_NAME,
            run_utc=dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat(),
            articles=len(result), use=result['use'].value_counts().to_dict(),
            reason=result.loc[result['use'] == 'exclude', 'reason'].value_counts().to_dict())
(OUT / 'filter_meta.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
print('저장:', (OUT / 'article_filter.csv').relative_to(ROOT), len(result), '줄')
print(json.dumps(meta, ensure_ascii=False, indent=2))

저장: source/news/filter/article_filter.csv 2725 줄
{
  "filter_version": "2026-11-11",
  "source": "source/news/snapshot_20261112_1127/news.sqlite3",
  "run_utc": "2026-11-12T02:33:13+00:00",
  "articles": 2725,
  "use": {
    "body": 2357,
    "exclude": 254,
    "summary": 114
  },
  "reason": {
    "status": 54,
    "media": 35,
    "sponsored": 33,
    "lifestyle": 29,
    "data_page": 27,
    "stale": 21,
    "opinion": 21,
    "promo": 14,
    "non_english": 10,
    "duplicate": 6,
    "people_roundup": 4
  }
}


## 8. 결과 확인

매체별 건수와 사유별 예시를 본다. 규칙이 실제 기사를 잘못 거르지 않았는지, 걸러야 할 글이 남지 않았는지 여기서 확인하고 4절 규칙을 고친다.

In [9]:
# ── 확인: 사유별 건수 ──
# 사유별 건수
summary = result.assign(사유=result['reason_ko'].replace('', '(사용)'))
summary.groupby(['use', '사유']).size().rename('기사 수').to_frame()

기사 수
use     사유                           
body    (사용)                     2357
exclude 매체 홍보·행사 안내                14
        본문 상태(수집 단계에서 쓰지 않기로 함)    54
        생활·커리어                     29
        수집 때 이미 오래된 기사             21
        시세·자료·라이브 페이지              27
        영상·팟캐스트·사진                 35
        영어가 아닌 기사                  10
        외부 기고                      21
        인사 동정 모음                    4
        중복                          6
        후원·광고                      33
summary (사용)                      114

In [10]:
# ── 확인: 매체별 건수 ──
# 매체별: 전체, 본문 사용, 제목·요약만, 제외, 미룸
by_source = (result.pivot_table(index='source', columns='use', values='article_id', aggfunc='size', fill_value=0)
             .reindex(columns=['body', 'summary', 'exclude', 'wait'], fill_value=0))
by_source.insert(0, 'total', by_source.sum(axis=1))
by_source.sort_values('total', ascending=False)

use,total,body,summary,exclude,wait
source,,,,,
CNBC,428,378,2,48,0
GamesIndustry.biz,109,106,0,3,0
LA Times,106,106,0,0,0
Canary Media,106,105,0,1,0
Fortune,94,68,0,26,0
FreightWaves,93,83,0,10,0
Maritime Executive,90,86,0,4,0
Oil & Gas Journal,82,59,0,23,0
Offshore Energy,67,67,0,0,0


In [11]:
# ── 확인: 규칙으로 제외한 기사 예 ──
# 규칙으로 제외한 기사 (사유별 최대 8건)
excluded = result[result['use'].eq('exclude') & ~result['reason'].isin(['status'])]
excluded.groupby('reason', group_keys=False).head(8)[['reason_ko', 'source', 'body_status', 'title', 'rule_matched']]

,reason_ko,source,body_status,title,rule_matched
0,수집 때 이미 오래된 기사,CBS News,ok,Nearly one in five say coronavirus has caused financial hardship,
1,수집 때 이미 오래된 기사,FreightWaves,ok,Project44 acquires ClearMetal to strengthen predictive tools,
2,수집 때 이미 오래된 기사,FreightWaves,ok,Seasonality pushing rejections and rates higher ahead of the Fourth,
3,영상·팟캐스트·사진,FreightWaves,ok,Warehouse cramming is about to begin — Freightonomics,tag:(^|\|)(videos?|podcasts?|tv & podcasts|galleries)(\||$)
4,수집 때 이미 오래된 기사,FreightWaves,ok,Exclusive: Central Freight Lines to shut down after 96 years,
5,수집 때 이미 오래된 기사,FreightWaves,ok,Why the Northeast is quietly running out of diesel,
6,수집 때 이미 오래된 기사,FreightWaves,ok,The ‘ingenious strategy’ behind most truckers’ least favorite week of the year: Intern...,
7,수집 때 이미 오래된 기사,CBS News,ok,"Tracking cost of cars, trucks, groceries as Trump announces new tariffs",
8,생활·커리어,Fortune,ok,Mary Barra still responds to ‘every single letter’ she gets by hand despite running $6...,tag:(^|\|)(success|workplace culture|future of work|c-suite|travel & leisure)(\||$)
9,수집 때 이미 오래된 기사,CNBC,ok,Top Wall Street analysts like these dividend stocks for steady income,


In [12]:
# ── 확인: 제목·요약만 쓰는 기사 예 ──
# 제목·요약만 쓰는 기사 예 (본문 상태별 최대 3건)
result[result['use'].eq('summary')].groupby('body_status', group_keys=False).head(3)[['body_status', 'source', 'title']]

,body_status,source,title
323,paywalled,OLED-Info,"Apple announces its first OLED laptop, the MacBook Pro with PROLED XDR Display"
331,paywalled,OLED-Info,"Tianma to expand its TM18 6-Gen flexible AMOLED line in Xiamen by 50%, to 48,000 month..."
502,blocked,Hotel Dive,Placemakr opens second apart-hotel property in Arizona
510,paywalled,OLED-Info,Apple is reportedly leaning towards sputtering for its next-generation HMO OLED backpl...
580,blocked,Hotel Dive,Sonesta brings Boyd Gaming casino resorts to distribution platform
616,blocked,For Construction Pros,CMAA Appoints Kevin Watkins President and CEO
1034,empty,World Nuclear News,The mothballed nuclear power plant
1246,empty,CNBC,This travel giant is caught in the AI selloff. Here's why I'm buying
2205,empty,CNBC,Nvidia hires law firm Wachtell Lipton to lead Jensen Huang affair probe
